# 🌌 3.2B PPC-GNN: Biologically Plausible LLM (Dual T4) — v2 (Final Stable)

This is the **Production Version**. It solves the 3.2B OOM error and the Gradient Disconnection bug.

### Key Architecture Features:
- **DEQ-PPC**: Analytical Gradient Bridge for MoE Experts.
- **Dynamic Convergence**: Early stopping saves ~40% compute on easy tokens.
- **FP16 Logic**: Compressed expert weights (3.2GB per GPU).
- **Sharding**: Balanced across 2x T4 GPUs.

In [ ]:
# ── Cell 1: Installation & Library Setup ──────────────────────────────────
import os, subprocess
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# 1. Force cleanup of conflicting CUDA links
!pip uninstall -y nvidia-nvjitlink-cu12 nvidia-curand-cu12 nvidia-cusolver-cu12 nvidia-cusparse-cu12 -q
os.system('pip install -q uv')

# 2. Dynamic CUDA Path Discovery (Kaggle Fix for bitsandbytes)
try:
    paths = subprocess.check_output(['find', '/usr/local', '-name', 'libcudart.so*'], text=True).splitlines()
    if paths:
        cuda_dir = os.path.dirname(paths[0])
        os.environ['LD_LIBRARY_PATH'] = f"{cuda_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
        print(f"✅ Injected CUDA path: {cuda_dir}")
except Exception as e:
    print(f"⚠️ CUDA discovery failed: {e}")

# 3. Install core dependencies + latest library from GitHub
# Pin numpy < 2.0.0 to prevent AttributeError in transformers/scikit-learn
!uv pip install --system --force-reinstall -q "numpy>=1.24.0,<2.0.0" bitsandbytes>=0.45.0 datasets transformers wandb tqdm git+https://github.com/ey3lock3r/EFV-nn.git

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils
from torch.utils.checkpoint import checkpoint
import bitsandbytes as bnb
import bitsandbytes.nn as bnb_nn
import wandb, gc
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
from kaggle_secrets import UserSecretsClient
import math, time

# Modular Import
from efv_nn.ppc_sharded import ShardedPPCGraphLLM

print(f"CUDA Available: {torch.cuda.is_available()} | Devices: {torch.cuda.device_count()}")


In [ ]:
# ── Cell 2: Secrets & Monitoring ────────────────
try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    WANDB_KEY = secrets.get_secret("WANDB_API_KEY")
    
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['WANDB_API_KEY'] = WANDB_KEY
    wandb.login(key=WANDB_KEY)
    print("✅ Logged into W&B and HF")
except Exception as e:
    print(f"⚠️ Secrets failure: {e}. Training will continue without W&B.")


In [ ]:
# ── Cell 3: Data Pipeline (FineWeb-Edu Streaming) ───────────────
def get_dataloader(tokenizer, batch_size=2, seq_len=256):
    ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
    def gen():
        buffer = []
        for ex in ds:
            buffer.extend(tokenizer(ex["text"])["input_ids"])
            while len(buffer) >= (batch_size * seq_len):
                yield torch.tensor(buffer[:batch_size * seq_len]).view(batch_size, seq_len)
                buffer = buffer[batch_size * seq_len:]
    return gen()


In [ ]:
# ── Cell 4: Training & High-Efficiency Checkpointing ─────────────
VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS = 128256, 1024, 24, 64
CKPT_PATH = "/kaggle/working/ppc_3b_pretrain.pt"
RESTART_FROM_SCRATCH = True # Set to False for normal resume

def save_checkpoint(model, step, path):
    gc.collect()
    torch.cuda.empty_cache()
    state_dict = model.state_dict()
    compressed_dict = {k: (v.half() if v.dtype == torch.float32 else v) for k, v in state_dict.items()}
    torch.save({'step': step, 'model_state': compressed_dict}, path)
    wandb.save(path)
    print(f"✅ Step {step} saved & compressed.")


def nuke_vram():
    import gc
    global model, opt, scaler, logits, loss
    for var in ['model', 'opt', 'scaler', 'logits', 'loss']:
        if var in globals():
            del globals()[var]
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.ipc_collect()
    print("☢️ Nuclear VRAM Reset Complete.")

def train():
    nuke_vram()

    import gc
    gc.collect()
    torch.cuda.empty_cache()
    # ENABLE JACOBIAN FIX
    model = ShardedPPCGraphLLM(VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS, use_jacobian=True)
    # TUNE LEARNING RATES (ADAMW FOR DECOUPLED WEIGHT DECAY)
    opt = bnb.optim.PagedAdamW8bit([
        {'params': model.embedding.parameters(), 'lr': 1e-4},
        {'params': model.layers.parameters(), 'lr': 1.5e-4}, 
        {'params': model.output_head.parameters(), 'lr': 1e-4},
        {'params': model.layer_norm.parameters(), 'lr': 1e-4}
    ])
    
    start_step = 0
    if os.path.exists(CKPT_PATH) and RESTART_FROM_SCRATCH:
        os.remove(CKPT_PATH)
        print("🔄 Restarting from scratch as requested.")
    
    if os.path.exists(CKPT_PATH):
        ckpt = torch.load(CKPT_PATH, map_location='cpu')
        loaded, current = ckpt['model_state'], model.state_dict()
        for k in current:
            if k in loaded:
                data = loaded[k]
                if torch.is_complex(current[k]) and not torch.is_complex(data): data = torch.view_as_complex(data.float())
                if 'experts_weight_real' in k: data = data.to(current[k].dtype)
                current[k].copy_(data)
        start_step = ckpt['step']
        print(f"🔄 Resumed Step {start_step}")

    wandb.init(project="ppc-3b-dynamic", name=f"ppc-3b-{time.strftime('%H%M')}")
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B", token=os.environ.get('HF_TOKEN'))
    dataloader = get_dataloader(tokenizer, batch_size=2, seq_len=256)
    
    for step, batch in enumerate(dataloader):
        if step < start_step: continue
        start = time.time()
        ids, targets = batch[:, :-1], batch[:, 1:].to(model.device1)
        
        opt.zero_grad()
        torch.cuda.empty_cache()
        # Autocast handled by PagedAdam8bit and manual unpacking
        with torch.amp.autocast('cuda'):
            logits, avg_iters = model(ids, local_iters=16)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
        
        # MANUALLY scale the loss down to avoid FP16 param inf crash completely
        scaled_loss = loss / 256.0
        scaled_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0 / 256.0)
        opt.step()
        
        if step % 5 == 0:
            wandb.log({"loss": loss.item(), "ppl": math.exp(min(loss.item(), 20)), "avg_ppc_iters": avg_iters, "step": step})
            print(f"Step {step} | Loss: {loss.item():.4f} | Iters: {avg_iters:.1f}/16.0")
            
        if (step + 1) % 100 == 0: save_checkpoint(model, step + 1, CKPT_PATH)
        if step % 50 == 0: torch.cuda.empty_cache()

train()


In [ ]:
# ── Cell 5: 🗣️ Generation Verification ──────────────────
def verify_generation(prompt="The fundamental principles of biology include"):
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B", token=os.environ.get('HF_TOKEN'))
    model = ShardedPPCGraphLLM(128256, 1024, 24, 64)
    if os.path.exists(CKPT_PATH):
        ckpt = torch.load(CKPT_PATH, map_location='cpu')
        model.load_state_dict(ckpt['model_state'], strict=False)
    inputs = tokenizer(prompt, return_tensors="pt")["input_ids"]
    output_ids = model.generate(inputs, max_new_tokens=50)
    print(f"\nPrompt: {prompt}\n\nOutput:\n{tokenizer.decode(output_ids[0], skip_special_tokens=True)}")

verify_generation()
